## Data Wrangling Project
### 2. Data Preparation  
Travis Letsch

In [1]:
import pandas as pd
import requests
import time
import numpy as np
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

<br>

## Overview 
This is the second of three notebooks for this project. This notebook cleans and combines the NUFORC data produced in the first notebook, then merges location data, population, and power plant data into the DataFrame. The resulting file, nuforc_complete.csv, is ready for analysis in the third notebook.

### NUFORC Data Prep

#### Import the CSV files and create DataFrames for each year

In [2]:
nuforc_2023 = pd.read_csv("nuforc_2023.csv", encoding="utf-8", sep=",")

# Add a year column to merge with population data later 
nuforc_2023["year"] = "2023"

nuforc_2023.head()

,occurred,city,state,country,shape,summary,year
0,01/31/2023 19:49,Gothenburg,NE,USA,Fireball,Saw trailblazer that moves very rapid and move...,2023
1,01/31/2023 19:15,Sulphur Springs,IN,USA,Circle,Fast spherical orange light,2023
2,01/31/2023 18:38,El Granada,CA,USA,Light,Image of these UFO's were capture with Nikon D...,2023
3,01/31/2023 14:30,Layton,UT,USA,Circle,Working outside with two dozen others looked u...,2023
4,01/31/2023 08:20,Larnaca,Larnaca District,Cyprus,Other,Something like drone,2023


<br>

In [3]:
nuforc_2024 = pd.read_csv("nuforc_2024.csv", encoding="utf-8", sep=",")

# Add a year column to merge with population data later 
nuforc_2024["year"] = "2024"

nuforc_2024.head()

,occurred,city,state,country,shape,summary,year
0,01/31/2024 23:56,Raleigh,NC,USA,Orb,Saw an orb looking craft over my house,2024
1,01/31/2024 19:30,Houston,TX,USA,Triangle,"It was big, triangular shaped and had multiple...",2024
2,01/31/2024 19:10,Texarkana,TX,USA,Triangle,Triangle shaped with dim lights on bottom,2024
3,01/31/2024 18:15,Surry,VA,USA,Other,Pyramid shape with lights on one side,2024
4,01/31/2024 03:12,La Salle,IL,USA,Sphere,Light Blue Sphere in the sky shining through m...,2024


<br>

In [4]:
nuforc_2025 = pd.read_csv("nuforc_2025.csv", encoding="utf-8", sep=",")

# Add a year column to merge with population data later 
nuforc_2025["year"] = "2025"

nuforc_2025.head()

,occurred,city,state,country,shape,summary,year
0,01/31/2025 23:45,Leicester,England,United Kingdom,Triangle,Felt like time stood still. Appeared to be 3 c...,2025
1,01/31/2025 20:20,Nokomis,FL,USA,Circle,Multiple objects around Venus over an hour lon...,2025
2,01/31/2025 17:50,Nacogdoches,TX,USA,Cylinder,"Cylinder object was traveling above tree line,...",2025
3,01/31/2025 15:51,San Diego,CA,USA,Cigar,A white pillar / Cigar shaped object moving in...,2025
4,01/31/2025 09:40,NaN,CA,USA,Cigar,Enormous cigar-shaped object hovering over Pal...,2025


<br>

#### Check columns for each DataFrame to ensure they match before vertically integrating them

In [ ]:
nuforc_2023_cols = nuforc_2023.columns
nuforc_2024_cols = nuforc_2024.columns
nuforc_2025_cols = nuforc_2025.columns

print(nuforc_2023_cols)
print(nuforc_2024_cols)
print(nuforc_2025_cols)

Index(['occurred', 'city', 'state', 'country', 'shape', 'summary', 'year'], dtype='object')
Index(['occurred', 'city', 'state', 'country', 'shape', 'summary', 'year'], dtype='object')
Index(['occurred', 'city', 'state', 'country', 'shape', 'summary', 'year'], dtype='object')


<br>

#### Concatenate the DataFrames

In [6]:
nuforc_combined = pd.concat([nuforc_2023, nuforc_2024, nuforc_2025], ignore_index=True)

# Check DataFrame records to ensure they match
print(f"nuforc_2023 records: {len(nuforc_2023)}")
print(f"nuforc_2024 records: {len(nuforc_2024)}")
print(f"nuforc_2025 records: {len(nuforc_2025)}")
print(f"nuforc_combined records: {len(nuforc_combined)}")

nuforc_2023 records: 4745
nuforc_2024 records: 4253
nuforc_2025 records: 4489
nuforc_combined records: 13487


<br>

#### Data Cleaning

In [7]:
# Check data types
nuforc_combined.dtypes

occurred    object
city        object
state       object
country     object
shape       object
summary     object
year        object
dtype: object

In [8]:
# Convert the occurred column to datetime and rename it
nuforc_combined["occurred"] = pd.to_datetime(
    nuforc_combined["occurred"],
    format='%m/%d/%Y %H:%M')

nuforc_combined = nuforc_combined.rename(columns={"occurred": "date_time"})

nuforc_combined.dtypes

date_time    datetime64[ns]
city                 object
state                object
country              object
shape                object
summary              object
year                 object
dtype: object

In [9]:
display(nuforc_combined)

,date_time,city,state,country,shape,summary,year
0,2023-01-31 19:49:00,Gothenburg,NE,USA,Fireball,Saw trailblazer that moves very rapid and move...,2023
1,2023-01-31 19:15:00,Sulphur Springs,IN,USA,Circle,Fast spherical orange light,2023
2,2023-01-31 18:38:00,El Granada,CA,USA,Light,Image of these UFO's were capture with Nikon D...,2023
3,2023-01-31 14:30:00,Layton,UT,USA,Circle,Working outside with two dozen others looked u...,2023
4,2023-01-31 08:20:00,Larnaca,Larnaca District,Cyprus,Other,Something like drone,2023
...,...,...,...,...,...,...,...
13482,2025-12-01 19:02:00,Gebele,NaN,Azerbaijan,Light,"Observed a light looking like a sattelite, but...",2025
13483,2025-12-01 18:15:00,Adwick le Street,England,United Kingdom,Oval,Light moving slowly. No sound at all,2025
13484,2025-12-01 17:45:00,Tolland,CT,USA,Light,Witnessed 3 blue lights move across the sky wi...,2025
13485,2025-12-01 13:00:00,Dayton,Western Australia,Australia,Unknown,Cloaked aircraft flew over my vehicle,2025


<br>

In [10]:
# Correct the record with the shape of CIrcle instead of Circle
nuforc_combined["shape"] = nuforc_combined["shape"].replace("CIrcle", "Circle")

In [11]:
# Create a variable to store data for sightings outside of the USA
foreign_data = nuforc_combined[nuforc_combined["country"] != "USA"].index

display(foreign_data)

Index([    4,    16,    27,    28,    40,    53,    74,    77,    89,   103,
       ...
       13450, 13452, 13459, 13461, 13465, 13474, 13479, 13482, 13483, 13485],
      dtype='int64', length=2668)

<br>

In [12]:
# Drop records for sightings outside of the USA
nuforc_combined = nuforc_combined.drop(foreign_data)

<br>

In [13]:
# Check for missing cities. The state and city are required to reverse geocode the county
missing_city = nuforc_combined[nuforc_combined["city"].isna()]

display(missing_city)
print("-" * 25)
print(f"Records with missing cities: {len(missing_city)}")

,date_time,city,state,country,shape,summary,year
99,2023-01-22 21:25:00,NaN,FL,USA,Light,"Over 30 min, observed a starlike object, movin...",2023
125,2023-01-19 23:45:00,NaN,AZ,USA,Cross,Size of a small commercial plane hovering like...,2023
140,2023-01-19 00:24:00,NaN,VA,USA,Cylinder,Tic tac shapes moving about the sky at unbelie...,2023
146,2023-01-18 04:00:00,NaN,NM,USA,Unknown,A Strange Light,2023
152,2023-01-17 19:25:00,NaN,CA,USA,Diamond,Diamond shaped thing in the sky.,2023
...,...,...,...,...,...,...,...
12834,2025-11-30 22:22:00,NaN,AL,USA,Circle,Craft goes steady with another moving erratically,2025
12928,2025-11-21 12:56:00,NaN,LA,USA,Egg,High up in the sky. Airplane high.,2025
13200,2025-11-03 16:12:00,NaN,PA,USA,Cigar,Aprox 40 to 50 mph as noted on my speedometer....,2025
13256,2025-12-26 23:59:00,NaN,CA,USA,Other,Black craft that prevented seeing stars,2025


-------------------------
Records with missing cities: 127


<br>

In [14]:
# Drop records with missing cities
nuforc_combined["city"] = nuforc_combined["city"].str.strip()
nuforc_combined = nuforc_combined.dropna(subset=["city"])

In [15]:
display(nuforc_combined)

,date_time,city,state,country,shape,summary,year
0,2023-01-31 19:49:00,Gothenburg,NE,USA,Fireball,Saw trailblazer that moves very rapid and move...,2023
1,2023-01-31 19:15:00,Sulphur Springs,IN,USA,Circle,Fast spherical orange light,2023
2,2023-01-31 18:38:00,El Granada,CA,USA,Light,Image of these UFO's were capture with Nikon D...,2023
3,2023-01-31 14:30:00,Layton,UT,USA,Circle,Working outside with two dozen others looked u...,2023
5,2023-01-30 21:15:00,Richmond,VA,USA,Circle,Unexplained movement of one object and 40 or s...,2023
...,...,...,...,...,...,...,...
13478,2025-12-01 22:00:00,Brewerton,NY,USA,Triangle,Using night vision & FLIR; captured black tria...,2025
13480,2025-12-01 21:00:00,Jamestown,NY,USA,Light,Two pilots observed three aircraft,2025
13481,2025-12-01 20:50:00,Rockaway Township,NJ,USA,Orb,White orb floating and suddenly dropping from sky,2025
13484,2025-12-01 17:45:00,Tolland,CT,USA,Light,Witnessed 3 blue lights move across the sky wi...,2025


<br>

In [16]:
# Check for missing states. The state and city are required to reverse geocode the county
missing_state = nuforc_combined[nuforc_combined["state"].isna()]

display(missing_state)
print("-" * 25)
print(f"Records with missing states: {len(missing_state)}")

,date_time,city,state,country,shape,summary,year


-------------------------
Records with missing states: 0


<br>

In [17]:
# Check unique values in the state column
nuforc_combined["state"].unique()

array(['NE', 'IN', 'CA', 'UT', 'VA', 'FL', 'CT', 'MN', 'CO', 'SC', 'NY',
       'MA', 'WA', 'NJ', 'OR', 'KY', 'NC', 'AL', 'PA', 'OH', 'MD', 'AZ',
       'NH', 'GA', 'NM', 'LA', 'TX', 'TN', 'WV', 'HI', 'MT', 'OK', 'MI',
       'MO', 'IA', 'MS', 'IL', 'NV', 'VT', 'DE', 'AK', 'WI', 'KS', 'AR',
       'DC', 'SD', 'ND', 'ID', 'WY', 'ME', '-', 'PR', 'RI', '0',
       'New York', 'VI', 'Montana', 'Wisconsin', 'Ohio'], dtype=object)

<br>

In [18]:
# Rename Montana to MT
nuforc_combined["state"] = nuforc_combined["state"].str.strip()
nuforc_combined["state"] = nuforc_combined["state"].replace("Montana", "MT")

In [19]:
# Remove states with an entry of "-"
nuforc_combined = nuforc_combined[nuforc_combined["state"] != "-"]

In [20]:
# Check unique values in the state column
nuforc_combined["state"].unique()

array(['NE', 'IN', 'CA', 'UT', 'VA', 'FL', 'CT', 'MN', 'CO', 'SC', 'NY',
       'MA', 'WA', 'NJ', 'OR', 'KY', 'NC', 'AL', 'PA', 'OH', 'MD', 'AZ',
       'NH', 'GA', 'NM', 'LA', 'TX', 'TN', 'WV', 'HI', 'MT', 'OK', 'MI',
       'MO', 'IA', 'MS', 'IL', 'NV', 'VT', 'DE', 'AK', 'WI', 'KS', 'AR',
       'DC', 'SD', 'ND', 'ID', 'WY', 'ME', 'PR', 'RI', '0', 'New York',
       'VI', 'Wisconsin', 'Ohio'], dtype=object)

In [21]:
display(nuforc_combined)

,date_time,city,state,country,shape,summary,year
0,2023-01-31 19:49:00,Gothenburg,NE,USA,Fireball,Saw trailblazer that moves very rapid and move...,2023
1,2023-01-31 19:15:00,Sulphur Springs,IN,USA,Circle,Fast spherical orange light,2023
2,2023-01-31 18:38:00,El Granada,CA,USA,Light,Image of these UFO's were capture with Nikon D...,2023
3,2023-01-31 14:30:00,Layton,UT,USA,Circle,Working outside with two dozen others looked u...,2023
5,2023-01-30 21:15:00,Richmond,VA,USA,Circle,Unexplained movement of one object and 40 or s...,2023
...,...,...,...,...,...,...,...
13478,2025-12-01 22:00:00,Brewerton,NY,USA,Triangle,Using night vision & FLIR; captured black tria...,2025
13480,2025-12-01 21:00:00,Jamestown,NY,USA,Light,Two pilots observed three aircraft,2025
13481,2025-12-01 20:50:00,Rockaway Township,NJ,USA,Orb,White orb floating and suddenly dropping from sky,2025
13484,2025-12-01 17:45:00,Tolland,CT,USA,Light,Witnessed 3 blue lights move across the sky wi...,2025


In [22]:
# Create a variable to store records from the Virgin Islands
vi_records = nuforc_combined[nuforc_combined["state"] == "VI"].index

print(vi_records)

Index([7611], dtype='int64')


In [23]:
# Drop records from the Virgin Islands
nuforc_combined = nuforc_combined.drop(vi_records)

<br>

In [24]:
# Create a variable to store records from Puerto Rico
pr_records = nuforc_combined[nuforc_combined["state"] == "PR"].index

print(pr_records)

Index([1731], dtype='int64')


In [25]:
# Drop records from Puerto Rico
nuforc_combined = nuforc_combined.drop(pr_records)

In [26]:
print(sorted(nuforc_combined["state"].unique()))

['0', 'AK', 'AL', 'AR', 'AZ', 'CA', 'CO', 'CT', 'DC', 'DE', 'FL', 'GA', 'HI', 'IA', 'ID', 'IL', 'IN', 'KS', 'KY', 'LA', 'MA', 'MD', 'ME', 'MI', 'MN', 'MO', 'MS', 'MT', 'NC', 'ND', 'NE', 'NH', 'NJ', 'NM', 'NV', 'NY', 'New York', 'OH', 'OK', 'OR', 'Ohio', 'PA', 'RI', 'SC', 'SD', 'TN', 'TX', 'UT', 'VA', 'VT', 'WA', 'WI', 'WV', 'WY', 'Wisconsin']


In [27]:
nuforc_combined.reset_index(drop=True, inplace=True)
nuforc_combined

,date_time,city,state,country,shape,summary,year
0,2023-01-31 19:49:00,Gothenburg,NE,USA,Fireball,Saw trailblazer that moves very rapid and move...,2023
1,2023-01-31 19:15:00,Sulphur Springs,IN,USA,Circle,Fast spherical orange light,2023
2,2023-01-31 18:38:00,El Granada,CA,USA,Light,Image of these UFO's were capture with Nikon D...,2023
3,2023-01-31 14:30:00,Layton,UT,USA,Circle,Working outside with two dozen others looked u...,2023
4,2023-01-30 21:15:00,Richmond,VA,USA,Circle,Unexplained movement of one object and 40 or s...,2023
...,...,...,...,...,...,...,...
10676,2025-12-01 22:00:00,Brewerton,NY,USA,Triangle,Using night vision & FLIR; captured black tria...,2025
10677,2025-12-01 21:00:00,Jamestown,NY,USA,Light,Two pilots observed three aircraft,2025
10678,2025-12-01 20:50:00,Rockaway Township,NJ,USA,Orb,White orb floating and suddenly dropping from sky,2025
10679,2025-12-01 17:45:00,Tolland,CT,USA,Light,Witnessed 3 blue lights move across the sky wi...,2025


<br>

## Location Data Prep

In [28]:
# Filter only unique city and state combinations
unique_locations = nuforc_combined[["city", "state"]].drop_duplicates().copy()

print(unique_locations.head())
print("-" * 25)
print(f"Unique city and state combinations: {len(unique_locations)}")

              city state
0       Gothenburg    NE
1  Sulphur Springs    IN
2       El Granada    CA
3           Layton    UT
4         Richmond    VA
-------------------------
Unique city and state combinations: 5789


<br>

In [29]:
# Import the uscities Excel file
us_cities_df = pd.read_excel("uscities.xlsx")

us_cities_df.head()

,city,city_ascii,state_id,state_name,county_fips,county_name,lat,lng,population,density,source,military,incorporated,timezone,ranking,zips,id
0,New York,New York,NY,New York,36081,Queens,40.6943,-73.9249,19268388,10943.7,shape,False,True,America/New_York,1,11229 11228 11226 11225 11224 11222 11221 1122...,1840034016
1,Los Angeles,Los Angeles,CA,California,6037,Los Angeles,34.1141,-118.4068,11984083,3165.7,shape,False,True,America/Los_Angeles,1,91367 90291 90293 90292 91316 91311 90035 9003...,1840020491
2,Chicago,Chicago,IL,Illinois,17031,Cook,41.8375,-87.6866,8609571,4590.3,shape,False,True,America/Chicago,1,60018 60649 60641 60640 60643 60642 60645 6064...,1840000494
3,Miami,Miami,FL,Florida,12086,Miami-Dade,25.7840,-80.2101,6391670,4791.1,shape,False,True,America/New_York,1,33128 33129 33125 33126 33127 33149 33144 3314...,1840015149
4,Houston,Houston,TX,Texas,48201,Harris,29.7860,-95.3885,6227666,1386.2,shape,False,True,America/Chicago,1,77069 77068 77061 77060 77063 77062 77065 7706...,1840020925


<br>

In [30]:
# Format the city and state columns in nuforc_combined to merge with the uscities dataset 
nuforc_combined["city_clean"] = (nuforc_combined["city"].str.lower().str.strip())

nuforc_combined["state_clean"] = (nuforc_combined["state"].str.upper().str.strip())

In [31]:
# Format the uscities dataset to match nuforc_combined 
us_cities_df["city_clean"] = us_cities_df["city"].str.lower().str.strip()
us_cities_df["state_clean"] = us_cities_df["state_id"].str.upper().str.strip()

In [32]:
# Drop periods and commas from the city_clean columns of each DataFrame
for df in [nuforc_combined, us_cities_df]:
    df["city_clean"] = (
        df["city_clean"]
        .str.replace(".", "", regex=False)
        .str.replace(",", "", regex=False)
        .str.strip()
    )

In [33]:
# Handle duplicate cities by keeping the largest population match 
us_cities_df = us_cities_df.sort_values("population", ascending=False)

us_cities_df = us_cities_df.drop_duplicates(
    subset=["city_clean", "state_clean"],
    keep="first")

In [34]:
# Merge the datasets, appending the lat/long from the uscities dataset to nuforc_combined
merged = nuforc_combined.merge(
    us_cities_df[["city_clean", "state_clean", "county_name", "lat", "lng", "population", "county_fips"]],
    on=["city_clean", "state_clean"],
    how="left")

In [35]:
merged.head()

,date_time,city,state,country,shape,summary,year,city_clean,state_clean,county_name,lat,lng,population,county_fips
0,2023-01-31 19:49:00,Gothenburg,NE,USA,Fireball,Saw trailblazer that moves very rapid and move...,2023,gothenburg,NE,Dawson,40.9242,-100.1540,3452.0,31047.0
1,2023-01-31 19:15:00,Sulphur Springs,IN,USA,Circle,Fast spherical orange light,2023,sulphur springs,IN,Henry,39.9991,-85.4392,454.0,18065.0
2,2023-01-31 18:38:00,El Granada,CA,USA,Light,Image of these UFO's were capture with Nikon D...,2023,el granada,CA,San Mateo,37.5134,-122.4657,5232.0,6081.0
3,2023-01-31 14:30:00,Layton,UT,USA,Circle,Working outside with two dozen others looked u...,2023,layton,UT,Davis,41.0770,-111.9620,83286.0,49011.0
4,2023-01-30 21:15:00,Richmond,VA,USA,Circle,Unexplained movement of one object and 40 or s...,2023,richmond,VA,Richmond,37.5295,-77.4756,1098457.0,51760.0


In [36]:
# Check for missing lat / long values after merging
missing_fips = merged["county_fips"].isna().sum()

display(missing_fips)

np.int64(1543)

<br>

In [37]:
# Many cities with missing values have spelling errors, are small towns, or contain descriptions of the area ("West of Saint Paul")
# Drop the records with missing coordinates 
merged = merged.dropna(subset=["county_fips"])

In [38]:
merged.to_csv("merged_locations_clean.csv", index=False)

In [39]:
# Rename the longitude column to lon 
merged = merged.rename(columns={"lng": "lon"})

# Change county_fips to integer data type
merged["county_fips"] = merged["county_fips"].astype(int)

In [40]:
print(len(merged))

9138


<br>

In [41]:
display(merged)

,date_time,city,state,country,shape,summary,year,city_clean,state_clean,county_name,lat,lon,population,county_fips
0,2023-01-31 19:49:00,Gothenburg,NE,USA,Fireball,Saw trailblazer that moves very rapid and move...,2023,gothenburg,NE,Dawson,40.9242,-100.1540,3452.0,31047
1,2023-01-31 19:15:00,Sulphur Springs,IN,USA,Circle,Fast spherical orange light,2023,sulphur springs,IN,Henry,39.9991,-85.4392,454.0,18065
2,2023-01-31 18:38:00,El Granada,CA,USA,Light,Image of these UFO's were capture with Nikon D...,2023,el granada,CA,San Mateo,37.5134,-122.4657,5232.0,6081
3,2023-01-31 14:30:00,Layton,UT,USA,Circle,Working outside with two dozen others looked u...,2023,layton,UT,Davis,41.0770,-111.9620,83286.0,49011
4,2023-01-30 21:15:00,Richmond,VA,USA,Circle,Unexplained movement of one object and 40 or s...,2023,richmond,VA,Richmond,37.5295,-77.4756,1098457.0,51760
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10674,2025-12-02 17:52:00,Bartlesville,OK,USA,Light,I walked outside to let my dog out and looked ...,2025,bartlesville,OK,Washington,36.7365,-95.9456,37878.0,40147
10675,2025-12-02 04:15:00,Cimarron,KS,USA,Unknown,A minimum of three lights that were alternatel...,2025,cimarron,KS,Gray,37.8097,-100.3453,1949.0,20069
10676,2025-12-01 22:00:00,Brewerton,NY,USA,Triangle,Using night vision & FLIR; captured black tria...,2025,brewerton,NY,Onondaga,43.2345,-76.1410,4330.0,36067
10677,2025-12-01 21:00:00,Jamestown,NY,USA,Light,Two pilots observed three aircraft,2025,jamestown,NY,Chautauqua,42.0976,-79.2367,28188.0,36013


In [42]:
# Add the leading zero to 4 digit FIPS codes
merged["county_fips"] = merged["county_fips"].astype(str).str.zfill(5)

merged.head()

,date_time,city,state,country,shape,summary,year,city_clean,state_clean,county_name,lat,lon,population,county_fips
0,2023-01-31 19:49:00,Gothenburg,NE,USA,Fireball,Saw trailblazer that moves very rapid and move...,2023,gothenburg,NE,Dawson,40.9242,-100.1540,3452.0,31047
1,2023-01-31 19:15:00,Sulphur Springs,IN,USA,Circle,Fast spherical orange light,2023,sulphur springs,IN,Henry,39.9991,-85.4392,454.0,18065
2,2023-01-31 18:38:00,El Granada,CA,USA,Light,Image of these UFO's were capture with Nikon D...,2023,el granada,CA,San Mateo,37.5134,-122.4657,5232.0,06081
3,2023-01-31 14:30:00,Layton,UT,USA,Circle,Working outside with two dozen others looked u...,2023,layton,UT,Davis,41.0770,-111.9620,83286.0,49011
4,2023-01-30 21:15:00,Richmond,VA,USA,Circle,Unexplained movement of one object and 40 or s...,2023,richmond,VA,Richmond,37.5295,-77.4756,1098457.0,51760


In [43]:
# Check for sightings with missing records in the county_name column
missing_county = merged[merged["county_name"].isna()]

display(missing_county)
print("-" * 25)
print(f"Records with missing counties: {len(missing_county)}")

,date_time,city,state,country,shape,summary,year,city_clean,state_clean,county_name,lat,lon,population,county_fips


-------------------------
Records with missing counties: 0


<br>

In [44]:
# Reset the index for the DataFrame
merged.reset_index(drop=True, inplace=True)

In [45]:
merged.columns

Index(['date_time', 'city', 'state', 'country', 'shape', 'summary', 'year',
       'city_clean', 'state_clean', 'county_name', 'lat', 'lon', 'population',
       'county_fips'],
      dtype='object')

<br>

In [46]:
# Drop the population column from merged
merged.drop(columns={"population"})

# Create a new DataFrame from filtered columns and update column names
nuforc_geo_combined = merged[["year", "date_time", "summary", "shape", "city_clean", "state_clean", "county_name", "lat", "lon", "county_fips"]]

nuforc_geo_combined = nuforc_geo_combined.rename(columns={"city_clean": "city", "state_clean": "state", "county_name": "county", "lat": "sighting_lat", "lon": "sighting_lon"})

nuforc_geo_combined.head()

,year,date_time,summary,shape,city,state,county,sighting_lat,sighting_lon,county_fips
0,2023,2023-01-31 19:49:00,Saw trailblazer that moves very rapid and move...,Fireball,gothenburg,NE,Dawson,40.9242,-100.1540,31047
1,2023,2023-01-31 19:15:00,Fast spherical orange light,Circle,sulphur springs,IN,Henry,39.9991,-85.4392,18065
2,2023,2023-01-31 18:38:00,Image of these UFO's were capture with Nikon D...,Light,el granada,CA,San Mateo,37.5134,-122.4657,06081
3,2023,2023-01-31 14:30:00,Working outside with two dozen others looked u...,Circle,layton,UT,Davis,41.0770,-111.9620,49011
4,2023,2023-01-30 21:15:00,Unexplained movement of one object and 40 or s...,Circle,richmond,VA,Richmond,37.5295,-77.4756,51760


<br>

In [47]:
# Save cleaning progress
nuforc_geo_combined.to_csv("nuforc_geo_combined.csv", index=False)

<br>

## Population Data Prep and Integration

In [48]:
# The Census population Excel file has merged rows and notes in the header and footer
county_pop_data = pd.read_excel("co-est2025-pop.xlsx",
    sheet_name="CO-EST2025-POP",
    skiprows=5,      # Skip the header rows and the record for the United States
    skipfooter=6,    # Skip the notes in the footer
    header=None)

display(county_pop_data)

,0,1,2,3,4,5,6,7
0,".Autauga County, Alabama",58805,58914,59207,59755,60502,61476,61920
1,".Baldwin County, Alabama",231778,233252,239450,246703,254193,261652,267761
2,".Barbour County, Alabama",25225,24958,24510,24744,24681,24520,24607
3,".Bibb County, Alabama",22285,22181,22346,21976,21881,22225,22262
4,".Blount County, Alabama",59125,59099,59036,59450,59771,60173,60156
...,...,...,...,...,...,...,...,...
3139,".Sweetwater County, Wyoming",42268,42194,41636,41319,41294,41350,41267
3140,".Teton County, Wyoming",23330,23381,23607,23324,23354,23345,23333
3141,".Uinta County, Wyoming",20450,20462,20691,20715,20693,20653,20728
3142,".Washakie County, Wyoming",7681,7659,7723,7738,7731,7637,7597


In [49]:
# Designate the columns for the DataFrame
county_pop_data.columns = [
    "county_state",
    "pop_2020_base",
    "2020",
    "2021",
    "2022",
    "2023",
    "2024",
    "2025"
]

county_pop_data.head()

,county_state,pop_2020_base,2020,2021,2022,2023,2024,2025
0,".Autauga County, Alabama",58805,58914,59207,59755,60502,61476,61920
1,".Baldwin County, Alabama",231778,233252,239450,246703,254193,261652,267761
2,".Barbour County, Alabama",25225,24958,24510,24744,24681,24520,24607
3,".Bibb County, Alabama",22285,22181,22346,21976,21881,22225,22262
4,".Blount County, Alabama",59125,59099,59036,59450,59771,60173,60156


In [50]:
# Clean the county and state field
# Remove leading dot from county names
county_pop_data["county_state"] = (
    county_pop_data["county_state"]
    .str.replace(".", "", n=1, regex=False)
    .str.strip()
)

# Split county and state
county_pop_data[["county", "state"]] = county_pop_data["county_state"].str.rsplit(
    ", ",
    n=1,
    expand=True)

county_pop_data.head()

,county_state,pop_2020_base,2020,2021,2022,2023,2024,2025,county,state
0,"Autauga County, Alabama",58805,58914,59207,59755,60502,61476,61920,Autauga County,Alabama
1,"Baldwin County, Alabama",231778,233252,239450,246703,254193,261652,267761,Baldwin County,Alabama
2,"Barbour County, Alabama",25225,24958,24510,24744,24681,24520,24607,Barbour County,Alabama
3,"Bibb County, Alabama",22285,22181,22346,21976,21881,22225,22262,Bibb County,Alabama
4,"Blount County, Alabama",59125,59099,59036,59450,59771,60173,60156,Blount County,Alabama


<br>

In [51]:
# Read in the county_fips xlsx file as a DataFrame
county_fips_data = pd.read_excel("county_fips.xlsx")

display(county_fips_data)

,County,State,State Abbreviation,County FIPS Code,State FIPS code,County Name Fragment,County Population,County % of State Population,State Population
0,Autauga County,Alabama,AL,1001,1,autauga-county-al,59285.0,0.011494,5157699
1,Baldwin County,Alabama,AL,1003,1,baldwin-county-al,239945.0,0.046522,5157699
2,Barbour County,Alabama,AL,1005,1,barbour-county-al,24757.0,0.004800,5157699
3,Bibb County,Alabama,AL,1007,1,bibb-county-al,22152.0,0.004295,5157699
4,Blount County,Alabama,AL,1009,1,blount-county-al,59292.0,0.011496,5157699
...,...,...,...,...,...,...,...,...,...
3237,Vega Baja Municipio,Puerto Rico,PR,72145,72,NaN,54058.0,0.016876,3203295
3238,Vieques Municipio,Puerto Rico,PR,72147,72,NaN,8147.0,0.002543,3203295
3239,Villalba Municipio,Puerto Rico,PR,72149,72,NaN,21778.0,0.006799,3203295
3240,Yabucoa Municipio,Puerto Rico,PR,72151,72,NaN,29868.0,0.009324,3203295


<br>

In [52]:
# Add the leading zero to 4 digit FIPS codes
county_fips_data["County FIPS Code"] = county_fips_data["County FIPS Code"].astype(str).str.zfill(5)

county_fips_data.head()

,County,State,State Abbreviation,County FIPS Code,State FIPS code,County Name Fragment,County Population,County % of State Population,State Population
0,Autauga County,Alabama,AL,01001,1,autauga-county-al,59285.0,0.011494,5157699
1,Baldwin County,Alabama,AL,01003,1,baldwin-county-al,239945.0,0.046522,5157699
2,Barbour County,Alabama,AL,01005,1,barbour-county-al,24757.0,0.004800,5157699
3,Bibb County,Alabama,AL,01007,1,bibb-county-al,22152.0,0.004295,5157699
4,Blount County,Alabama,AL,01009,1,blount-county-al,59292.0,0.011496,5157699


<br>

In [53]:
# Filter the county_fips_data DataFrame
county_fips_data = county_fips_data[["County", "State", "County FIPS Code"]]

county_fips_data.head()

,County,State,County FIPS Code
0,Autauga County,Alabama,01001
1,Baldwin County,Alabama,01003
2,Barbour County,Alabama,01005
3,Bibb County,Alabama,01007
4,Blount County,Alabama,01009


<br>

In [54]:
# Standardize the case and spacing for each DataFrame
county_pop_data["County_clean"] = (
    county_pop_data["county"]
    .astype(str)
    .str.replace("-", " ", regex=False)
    .str.strip()
    .str.lower()
)

county_pop_data["State_clean"] = (
    county_pop_data["state"]
    .astype(str)
    .str.replace("-", " ", regex=False)
    .str.strip()
    .str.lower()
)

county_fips_data["County_clean"] = (
    county_fips_data["County"]
    .astype(str)
    .str.replace("-", " ", regex=False)
    .str.strip()
    .str.lower()
)

county_fips_data["State_clean"] = (
    county_fips_data["State"]
    .astype(str)
    .str.replace("-", " ", regex=False)
    .str.strip()
    .str.lower()
)

In [55]:
# Drop county, parish, and periods from the name in each County column
for df in [county_pop_data, county_fips_data]:
    df["County_clean"] = (
        df["County_clean"]
        .str.replace(".", "", regex=False)
        .str.replace(" county", "", regex=False)
        .str.replace(" parish", "", regex=False)
        .str.replace(" ", "", regex=False)
        .str.strip()
    )

In [56]:
# Append the FIPS codes to the population data using the county and state as the key
county_pop_fips = pd.merge(county_pop_data, county_fips_data, on=["County_clean", "State_clean"], how="left")

display(county_pop_fips)

,county_state,pop_2020_base,2020,2021,2022,2023,2024,2025,county,state,County_clean,State_clean,County,State,County FIPS Code
0,"Autauga County, Alabama",58805,58914,59207,59755,60502,61476,61920,Autauga County,Alabama,autauga,alabama,Autauga County,Alabama,01001
1,"Baldwin County, Alabama",231778,233252,239450,246703,254193,261652,267761,Baldwin County,Alabama,baldwin,alabama,Baldwin County,Alabama,01003
2,"Barbour County, Alabama",25225,24958,24510,24744,24681,24520,24607,Barbour County,Alabama,barbour,alabama,Barbour County,Alabama,01005
3,"Bibb County, Alabama",22285,22181,22346,21976,21881,22225,22262,Bibb County,Alabama,bibb,alabama,Bibb County,Alabama,01007
4,"Blount County, Alabama",59125,59099,59036,59450,59771,60173,60156,Blount County,Alabama,blount,alabama,Blount County,Alabama,01009
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3139,"Sweetwater County, Wyoming",42268,42194,41636,41319,41294,41350,41267,Sweetwater County,Wyoming,sweetwater,wyoming,Sweetwater County,Wyoming,56037
3140,"Teton County, Wyoming",23330,23381,23607,23324,23354,23345,23333,Teton County,Wyoming,teton,wyoming,Teton County,Wyoming,56039
3141,"Uinta County, Wyoming",20450,20462,20691,20715,20693,20653,20728,Uinta County,Wyoming,uinta,wyoming,Uinta County,Wyoming,56041
3142,"Washakie County, Wyoming",7681,7659,7723,7738,7731,7637,7597,Washakie County,Wyoming,washakie,wyoming,Washakie County,Wyoming,56043


<br>

In [57]:
# Rename columns after merging
county_pop_fips = county_pop_fips.rename(columns={"County_clean": "County", "State_clean": "State"})

display(county_pop_fips)

,county_state,pop_2020_base,2020,2021,2022,2023,2024,2025,county,state,County,State,County,State,County FIPS Code
0,"Autauga County, Alabama",58805,58914,59207,59755,60502,61476,61920,Autauga County,Alabama,autauga,alabama,Autauga County,Alabama,01001
1,"Baldwin County, Alabama",231778,233252,239450,246703,254193,261652,267761,Baldwin County,Alabama,baldwin,alabama,Baldwin County,Alabama,01003
2,"Barbour County, Alabama",25225,24958,24510,24744,24681,24520,24607,Barbour County,Alabama,barbour,alabama,Barbour County,Alabama,01005
3,"Bibb County, Alabama",22285,22181,22346,21976,21881,22225,22262,Bibb County,Alabama,bibb,alabama,Bibb County,Alabama,01007
4,"Blount County, Alabama",59125,59099,59036,59450,59771,60173,60156,Blount County,Alabama,blount,alabama,Blount County,Alabama,01009
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3139,"Sweetwater County, Wyoming",42268,42194,41636,41319,41294,41350,41267,Sweetwater County,Wyoming,sweetwater,wyoming,Sweetwater County,Wyoming,56037
3140,"Teton County, Wyoming",23330,23381,23607,23324,23354,23345,23333,Teton County,Wyoming,teton,wyoming,Teton County,Wyoming,56039
3141,"Uinta County, Wyoming",20450,20462,20691,20715,20693,20653,20728,Uinta County,Wyoming,uinta,wyoming,Uinta County,Wyoming,56041
3142,"Washakie County, Wyoming",7681,7659,7723,7738,7731,7637,7597,Washakie County,Wyoming,washakie,wyoming,Washakie County,Wyoming,56043


<br>

#### Create DataFrames for 2023, 2024, 2025

In [58]:
# Create 2023 DataFrame
county_pop_2023 = pd.DataFrame(county_pop_fips[["County", "State", "County FIPS Code", "2023"]])

# Add year column to merge
county_pop_2023["year"] = "2023"

county_pop_2023.head()

,County,County,State,State,County FIPS Code,2023,year
0,autauga,Autauga County,alabama,Alabama,01001,60502,2023
1,baldwin,Baldwin County,alabama,Alabama,01003,254193,2023
2,barbour,Barbour County,alabama,Alabama,01005,24681,2023
3,bibb,Bibb County,alabama,Alabama,01007,21881,2023
4,blount,Blount County,alabama,Alabama,01009,59771,2023


<br>

In [59]:
# Create 2024 DataFrame
county_pop_2024 = pd.DataFrame(county_pop_fips[["County", "State", "County FIPS Code", "2024"]])

# Add year column to merge
county_pop_2024["year"] = "2024"

county_pop_2024.head()

,County,County,State,State,County FIPS Code,2024,year
0,autauga,Autauga County,alabama,Alabama,01001,61476,2024
1,baldwin,Baldwin County,alabama,Alabama,01003,261652,2024
2,barbour,Barbour County,alabama,Alabama,01005,24520,2024
3,bibb,Bibb County,alabama,Alabama,01007,22225,2024
4,blount,Blount County,alabama,Alabama,01009,60173,2024


<br>

In [60]:
# Create 2025 DataFrame
county_pop_2025 = pd.DataFrame(county_pop_fips[["County", "State", "County FIPS Code", "2025"]])

# Add year column to merge
county_pop_2025["year"] = "2025"

county_pop_2025.head()

,County,County,State,State,County FIPS Code,2025,year
0,autauga,Autauga County,alabama,Alabama,01001,61920,2025
1,baldwin,Baldwin County,alabama,Alabama,01003,267761,2025
2,barbour,Barbour County,alabama,Alabama,01005,24607,2025
3,bibb,Bibb County,alabama,Alabama,01007,22262,2025
4,blount,Blount County,alabama,Alabama,01009,60156,2025


<br>

In [61]:
# Rename County FIPS Code to fips_combined to use as a merge key, and rename the numeric year to population for each year of data
county_pop_2023 = county_pop_2023.rename(columns={"County FIPS Code": "county_fips", "2023": "population"})
county_pop_2024 = county_pop_2024.rename(columns={"County FIPS Code": "county_fips", "2024": "population"})
county_pop_2025 = county_pop_2025.rename(columns={"County FIPS Code": "county_fips", "2025": "population"})

# Filter the columns we'll use in the merged DataFrame 
county_pop_2023 = county_pop_2023[["county_fips", "population", "year"]]
county_pop_2024 = county_pop_2024[["county_fips", "population", "year"]]
county_pop_2025 = county_pop_2025[["county_fips", "population", "year"]]

<br>

#### Vertically merge each year of data before exporting

In [62]:
# Ensure columns match
print(county_pop_2023.columns)
print(county_pop_2024.columns)
print(county_pop_2025.columns)

Index(['county_fips', 'population', 'year'], dtype='object')
Index(['county_fips', 'population', 'year'], dtype='object')
Index(['county_fips', 'population', 'year'], dtype='object')


<br>

In [63]:
combined_population_data = pd.concat([county_pop_2023, county_pop_2024, county_pop_2025], ignore_index=True)

display(combined_population_data)

,county_fips,population,year
0,01001,60502,2023
1,01003,254193,2023
2,01005,24681,2023
3,01007,21881,2023
4,01009,59771,2023
...,...,...,...
9427,56037,41267,2025
9428,56039,23333,2025
9429,56041,20728,2025
9430,56043,7597,2025


<br>

In [64]:
# To ensure the leading 0 in the fips_combined column is kept
combined_population_data["county_fips"] = combined_population_data["county_fips"].astype(str).str.zfill(5)

# Export combined population DataFrame to a CSV file to save progress
combined_population_data.to_csv("combined_population_data.csv", index=False)

<br>

In [65]:
# Merge the combined population data to nuforc_geo_combined
nuforc_population = pd.merge(nuforc_geo_combined, combined_population_data, on=["county_fips", "year"], how="left")

display(nuforc_population)

,year,date_time,summary,shape,city,state,county,sighting_lat,sighting_lon,county_fips,population
0,2023,2023-01-31 19:49:00,Saw trailblazer that moves very rapid and move...,Fireball,gothenburg,NE,Dawson,40.9242,-100.1540,31047,24460.0
1,2023,2023-01-31 19:15:00,Fast spherical orange light,Circle,sulphur springs,IN,Henry,39.9991,-85.4392,18065,48887.0
2,2023,2023-01-31 18:38:00,Image of these UFO's were capture with Nikon D...,Light,el granada,CA,San Mateo,37.5134,-122.4657,06081,733949.0
3,2023,2023-01-31 14:30:00,Working outside with two dozen others looked u...,Circle,layton,UT,Davis,41.0770,-111.9620,49011,374645.0
4,2023,2023-01-30 21:15:00,Unexplained movement of one object and 40 or s...,Circle,richmond,VA,Richmond,37.5295,-77.4756,51760,230804.0
...,...,...,...,...,...,...,...,...,...,...,...
9133,2025,2025-12-02 17:52:00,I walked outside to let my dog out and looked ...,Light,bartlesville,OK,Washington,36.7365,-95.9456,40147,54037.0
9134,2025,2025-12-02 04:15:00,A minimum of three lights that were alternatel...,Unknown,cimarron,KS,Gray,37.8097,-100.3453,20069,5652.0
9135,2025,2025-12-01 22:00:00,Using night vision & FLIR; captured black tria...,Triangle,brewerton,NY,Onondaga,43.2345,-76.1410,36067,466584.0
9136,2025,2025-12-01 21:00:00,Two pilots observed three aircraft,Light,jamestown,NY,Chautauqua,42.0976,-79.2367,36013,124126.0


## Nuclear Plant Data Prep and Integration

In [66]:
nuclear_plants = pd.read_csv("global-power-plants.csv", encoding="utf-8", sep=",")

nuclear_plants.head()

,rowid,country,country_long,name,gppd_idnr,capacity_mw,latitude,longitude,primary_fuel,other_fuel1,...,geolocation_source,wepp_id,year_of_capacity_data,generation_gwh_2013,generation_gwh_2014,generation_gwh_2015,generation_gwh_2016,generation_gwh_2017,generation_data_source,estimated_generation_gwh
0,32367,USA,United States of America,Turkey Point,USA0000621,2861.2,25.4356,-80.3308,Nuclear,Oil,...,U.S. Energy Information Administration,20282|20284,2017,17871.732958,18371.295983,20337.856061,20506.629986,18022.696956,U.S. Energy Information Administration,NaN
1,31835,USA,United States of America,St Lucie,USA0006045,2160.0,27.3486,-80.2464,Nuclear,NaN,...,U.S. Energy Information Administration,20281,2017,15619.911000,15817.574000,15075.264000,15586.562000,16427.920000,U.S. Energy Information Administration,NaN
2,31732,USA,United States of America,South Texas Project,USA0006251,2708.6,28.7950,-96.0481,Nuclear,NaN,...,U.S. Energy Information Administration,22406,2017,17827.856000,20651.667000,19400.553000,21694.303000,21581.476000,U.S. Energy Information Administration,NaN
3,32796,USA,United States of America,Waterford 3,USA0004270,1199.8,29.9953,-90.4716,Nuclear,NaN,...,U.S. Energy Information Administration,25138,2017,9367.751000,9156.336000,8384.741000,10096.757000,8377.463000,U.S. Energy Information Administration,NaN
4,30968,USA,United States of America,River Bend,USA0006462,1035.9,30.7570,-91.3327,Nuclear,NaN,...,U.S. Energy Information Administration,19601,2017,7586.643000,8154.994000,6915.995000,7055.069000,7032.282000,U.S. Energy Information Administration,NaN


<br>

In [67]:
# Use the Census Bureau API to reverse geocode latitude and longitude to a state and county
def get_location_info(lat, lon):
    url = f"https://geo.fcc.gov/api/census/block/find?latitude={lat}&longitude={lon}&format=json"
    response = requests.get(url).json()
    
    try:
        county_name = response['County']['name']
        county_fips = response['County']['FIPS']
        
        state_name = response['State']['name']
        state_code = response['State']['code']
        state_fips = response['State']['FIPS']
        
        return county_name, county_fips, state_name, state_code, state_fips
    
    except KeyError:
        return None, None, None, None, None

In [68]:
nuclear_plants[['county', 'county_fips', 'state', 'state_code', 'state_fips']] = (
    nuclear_plants.apply(lambda row: get_location_info(row['latitude'], row['longitude']), axis=1, result_type='expand')
)

In [69]:
display(nuclear_plants)

,rowid,country,country_long,name,gppd_idnr,capacity_mw,latitude,longitude,primary_fuel,other_fuel1,...,generation_gwh_2015,generation_gwh_2016,generation_gwh_2017,generation_data_source,estimated_generation_gwh,county,county_fips,state,state_code,state_fips
0,32367,USA,United States of America,Turkey Point,USA0000621,2861.2,25.4356,-80.3308,Nuclear,Oil,...,20337.856061,20506.629986,18022.696956,U.S. Energy Information Administration,NaN,Miami-Dade County,12086,Florida,FL,12
1,31835,USA,United States of America,St Lucie,USA0006045,2160.0,27.3486,-80.2464,Nuclear,NaN,...,15075.264000,15586.562000,16427.920000,U.S. Energy Information Administration,NaN,St. Lucie County,12111,Florida,FL,12
2,31732,USA,United States of America,South Texas Project,USA0006251,2708.6,28.7950,-96.0481,Nuclear,NaN,...,19400.553000,21694.303000,21581.476000,U.S. Energy Information Administration,NaN,Matagorda County,48321,Texas,TX,48
3,32796,USA,United States of America,Waterford 3,USA0004270,1199.8,29.9953,-90.4716,Nuclear,NaN,...,8384.741000,10096.757000,8377.463000,U.S. Energy Information Administration,NaN,St. Charles Parish,22089,Louisiana,LA,22
4,30968,USA,United States of America,River Bend,USA0006462,1035.9,30.7570,-91.3327,Nuclear,NaN,...,6915.995000,7055.069000,7032.282000,U.S. Energy Information Administration,NaN,West Feliciana Parish,22125,Louisiana,LA,22
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
56,28399,USA,United States of America,James A Fitzpatrick,USA0006110,883.3,43.5214,-76.4084,Nuclear,NaN,...,7382.237000,5874.126000,6174.224000,U.S. Energy Information Administration,NaN,Oswego County,36075,New York,NY,36
57,30582,USA,United States of America,Point Beach Nuclear Plant,USA0004046,1311.0,44.2806,-87.5369,Nuclear,Oil,...,10008.039000,10151.411000,9648.848000,U.S. Energy Information Administration,NaN,Manitowoc County,55071,Wisconsin,WI,55
58,30657,USA,United States of America,Prairie Island,USA0001925,1186.2,44.6220,-92.6333,Nuclear,NaN,...,7374.711000,8263.058000,8744.731000,U.S. Energy Information Administration,NaN,Goodhue County,27049,Minnesota,MN,27
59,29581,USA,United States of America,Monticello Nuclear Facility,USA0001922,685.0,45.3338,-93.8493,Nuclear,NaN,...,4663.895000,5597.758000,5159.620000,U.S. Energy Information Administration,NaN,Wright County,27171,Minnesota,MN,27


<br>

In [70]:
# Filter nuclear_plants before exporting
nuclear_plants = nuclear_plants[["name", "latitude", "longitude", "county", "state", "county_fips"]]

display(nuclear_plants)

,name,latitude,longitude,county,state,county_fips
0,Turkey Point,25.4356,-80.3308,Miami-Dade County,Florida,12086
1,St Lucie,27.3486,-80.2464,St. Lucie County,Florida,12111
2,South Texas Project,28.7950,-96.0481,Matagorda County,Texas,48321
3,Waterford 3,29.9953,-90.4716,St. Charles Parish,Louisiana,22089
4,River Bend,30.7570,-91.3327,West Feliciana Parish,Louisiana,22125
...,...,...,...,...,...,...
56,James A Fitzpatrick,43.5214,-76.4084,Oswego County,New York,36075
57,Point Beach Nuclear Plant,44.2806,-87.5369,Manitowoc County,Wisconsin,55071
58,Prairie Island,44.6220,-92.6333,Goodhue County,Minnesota,27049
59,Monticello Nuclear Facility,45.3338,-93.8493,Wright County,Minnesota,27171


<br>

In [71]:
# Rename columns in preparation to merge DataFrames
nuclear_plants = nuclear_plants.rename(columns=
    {"name": "plant_name", "latitude": "plant_latitude", "longitude": "plant_longitude"})

In [72]:
# Filter the columns we'll use in the merged DataFrame
nuclear_plants = nuclear_plants[["plant_name", "plant_latitude", "plant_longitude", "county_fips"]]

display(nuclear_plants)

,plant_name,plant_latitude,plant_longitude,county_fips
0,Turkey Point,25.4356,-80.3308,12086
1,St Lucie,27.3486,-80.2464,12111
2,South Texas Project,28.7950,-96.0481,48321
3,Waterford 3,29.9953,-90.4716,22089
4,River Bend,30.7570,-91.3327,22125
...,...,...,...,...
56,James A Fitzpatrick,43.5214,-76.4084,36075
57,Point Beach Nuclear Plant,44.2806,-87.5369,55071
58,Prairie Island,44.6220,-92.6333,27049
59,Monticello Nuclear Facility,45.3338,-93.8493,27171


<br>

In [73]:
# To ensure the leading 0 in the county_fips column is kept
nuclear_plants["county_fips"] = nuclear_plants["county_fips"].astype(str).str.zfill(5)

# Export nuclear_plants to a CSV file to save progress
nuclear_plants.to_csv("nuclear_plant_data.csv", index=False)

In [74]:
# Aggregate plants per county, as some counties have more than 1
nuclear_grouped = nuclear_plants.groupby("county_fips").agg({
    "plant_name": lambda x: ", ".join(x),
    "plant_latitude": "mean",
    "plant_longitude": "mean",
    "county_fips": "count"
}).rename(columns={"county_fips": "num_plants"}).reset_index()

In [75]:
# Merge the nuclear plant data with the nuforc_population DataFrame using county_fips as the key
nuforc_complete = pd.merge(nuforc_population, nuclear_plants, on=["county_fips"], how="left")

display(nuforc_complete)

,year,date_time,summary,shape,city,state,county,sighting_lat,sighting_lon,county_fips,population,plant_name,plant_latitude,plant_longitude
0,2023,2023-01-31 19:49:00,Saw trailblazer that moves very rapid and move...,Fireball,gothenburg,NE,Dawson,40.9242,-100.1540,31047,24460.0,NaN,NaN,NaN
1,2023,2023-01-31 19:15:00,Fast spherical orange light,Circle,sulphur springs,IN,Henry,39.9991,-85.4392,18065,48887.0,NaN,NaN,NaN
2,2023,2023-01-31 18:38:00,Image of these UFO's were capture with Nikon D...,Light,el granada,CA,San Mateo,37.5134,-122.4657,06081,733949.0,NaN,NaN,NaN
3,2023,2023-01-31 14:30:00,Working outside with two dozen others looked u...,Circle,layton,UT,Davis,41.0770,-111.9620,49011,374645.0,NaN,NaN,NaN
4,2023,2023-01-30 21:15:00,Unexplained movement of one object and 40 or s...,Circle,richmond,VA,Richmond,37.5295,-77.4756,51760,230804.0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9147,2025,2025-12-02 17:52:00,I walked outside to let my dog out and looked ...,Light,bartlesville,OK,Washington,36.7365,-95.9456,40147,54037.0,NaN,NaN,NaN
9148,2025,2025-12-02 04:15:00,A minimum of three lights that were alternatel...,Unknown,cimarron,KS,Gray,37.8097,-100.3453,20069,5652.0,NaN,NaN,NaN
9149,2025,2025-12-01 22:00:00,Using night vision & FLIR; captured black tria...,Triangle,brewerton,NY,Onondaga,43.2345,-76.1410,36067,466584.0,NaN,NaN,NaN
9150,2025,2025-12-01 21:00:00,Two pilots observed three aircraft,Light,jamestown,NY,Chautauqua,42.0976,-79.2367,36013,124126.0,NaN,NaN,NaN


<br>

In [76]:
# Define a function to check if the county has a power plant then return a 1 or 0 
def plant_check(plant):
    if pd.notna(plant):
        return 1
    else:
        return 0

In [77]:
# Populate a new column by applying the plant_check function
nuforc_complete["plant_in_county"] = nuforc_complete["plant_name"].apply(plant_check)

<br>

#### Calculate sightings per 100k population for analysis

In [78]:
# Create a new DataFrame from nuforc_complete columns
county_pop_sightings = pd.DataFrame(nuforc_complete[["county_fips", "population", "year", "county", "state", "plant_in_county"]])

display(county_pop_sightings)

,county_fips,population,year,county,state,plant_in_county
0,31047,24460.0,2023,Dawson,NE,0
1,18065,48887.0,2023,Henry,IN,0
2,06081,733949.0,2023,San Mateo,CA,0
3,49011,374645.0,2023,Davis,UT,0
4,51760,230804.0,2023,Richmond,VA,0
...,...,...,...,...,...,...
9147,40147,54037.0,2025,Washington,OK,0
9148,20069,5652.0,2025,Gray,KS,0
9149,36067,466584.0,2025,Onondaga,NY,0
9150,36013,124126.0,2025,Chautauqua,NY,0


<br>

In [79]:
# Calculate sightings per county, then sightings per 100k population
# The count of county_fips is used to determine the number of sightings in a particular county
county_counts = (county_pop_sightings.groupby(["county_fips", "year"])
                 .agg(sightings= ("county_fips", "size"), population= ("population", "first"), county=("county", "first"), state=("state", "first"),
                     plant_in_county=("plant_in_county", "first")))

county_counts.reset_index()

display(county_counts)

sightings  population      county state  plant_in_county
county_fips year                                                          
01001       2024          1     61476.0     Autauga    AL                0
01003       2023          5    254193.0     Baldwin    AL                0
            2024          2    261652.0     Baldwin    AL                0
            2025          4    267761.0     Baldwin    AL                0
01005       2023          1     24681.0     Barbour    AL                0
...                     ...         ...         ...   ...              ...
56031       2023          1      8604.0      Platte    WY                0
            2025          1      8634.0      Platte    WY                0
56033       2025          2     33241.0    Sheridan    WY                0
56037       2023          2     41294.0  Sweetwater    WY                0
56039       2024          1     23345.0       Teton    WY                0

[3241 rows x 5 columns]

<br>

In [80]:
# Calculate the sightings per 100k population
county_counts["sightings_per_100k"] = (county_counts["sightings"] / county_counts["population"]) * 100000

county_counts = county_counts.reset_index()

display(county_counts)

,county_fips,year,sightings,population,county,state,plant_in_county,sightings_per_100k
0,01001,2024,1,61476.0,Autauga,AL,0,1.626651
1,01003,2023,5,254193.0,Baldwin,AL,0,1.967009
2,01003,2024,2,261652.0,Baldwin,AL,0,0.764374
3,01003,2025,4,267761.0,Baldwin,AL,0,1.493870
4,01005,2023,1,24681.0,Barbour,AL,0,4.051700
...,...,...,...,...,...,...,...,...
3236,56031,2023,1,8604.0,Platte,WY,0,11.622501
3237,56031,2025,1,8634.0,Platte,WY,0,11.582117
3238,56033,2025,2,33241.0,Sheridan,WY,0,6.016666
3239,56037,2023,2,41294.0,Sweetwater,WY,0,4.843319


<br>

In [81]:
# Merge the sightings and sightings_per_100k columns to nuforc_complete 
nuforc_complete = nuforc_complete.merge(county_counts[["sightings", "sightings_per_100k", "county_fips", "year"]],
    on=["county_fips", "year"],
    how="left")

In [82]:
# Drop records without a sightings_per_100k value, as they lacked population data
nuforc_complete = nuforc_complete.dropna(subset=["sightings_per_100k"])

In [83]:
# Rename sightings to county_sightings
nuforc_complete = nuforc_complete.rename(columns={"sightings": "county_sightings"})

display(nuforc_complete)

,year,date_time,summary,shape,city,state,county,sighting_lat,sighting_lon,county_fips,population,plant_name,plant_latitude,plant_longitude,plant_in_county,county_sightings,sightings_per_100k
0,2023,2023-01-31 19:49:00,Saw trailblazer that moves very rapid and move...,Fireball,gothenburg,NE,Dawson,40.9242,-100.1540,31047,24460.0,NaN,NaN,NaN,0,1,4.088307
1,2023,2023-01-31 19:15:00,Fast spherical orange light,Circle,sulphur springs,IN,Henry,39.9991,-85.4392,18065,48887.0,NaN,NaN,NaN,0,1,2.045534
2,2023,2023-01-31 18:38:00,Image of these UFO's were capture with Nikon D...,Light,el granada,CA,San Mateo,37.5134,-122.4657,06081,733949.0,NaN,NaN,NaN,0,4,0.544997
3,2023,2023-01-31 14:30:00,Working outside with two dozen others looked u...,Circle,layton,UT,Davis,41.0770,-111.9620,49011,374645.0,NaN,NaN,NaN,0,5,1.334597
4,2023,2023-01-30 21:15:00,Unexplained movement of one object and 40 or s...,Circle,richmond,VA,Richmond,37.5295,-77.4756,51760,230804.0,NaN,NaN,NaN,0,4,1.733072
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9147,2025,2025-12-02 17:52:00,I walked outside to let my dog out and looked ...,Light,bartlesville,OK,Washington,36.7365,-95.9456,40147,54037.0,NaN,NaN,NaN,0,2,3.701168
9148,2025,2025-12-02 04:15:00,A minimum of three lights that were alternatel...,Unknown,cimarron,KS,Gray,37.8097,-100.3453,20069,5652.0,NaN,NaN,NaN,0,1,17.692852
9149,2025,2025-12-01 22:00:00,Using night vision & FLIR; captured black tria...,Triangle,brewerton,NY,Onondaga,43.2345,-76.1410,36067,466584.0,NaN,NaN,NaN,0,3,0.642971
9150,2025,2025-12-01 21:00:00,Two pilots observed three aircraft,Light,jamestown,NY,Chautauqua,42.0976,-79.2367,36013,124126.0,NaN,NaN,NaN,0,1,0.805633


In [84]:
# Export the merged DataFrame to save progress
nuforc_complete.to_csv("nuforc_complete.csv", index=False)